In [6]:
# === Cell 1: imports + functions ===
from pathlib import Path
import gc,os,shutil,fiona,rasterio,time
from datetime import datetime
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from rasterio.transform import rowcol
from rasterio.windows import Window
from scipy.ndimage import uniform_filter, binary_dilation
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRFRegressor
from rasterio.warp import transform as crs_transform

In [2]:
# === Cell 2: make bathy raster and grid ===
def make_bath_rast(shp, template_tif, out_dir, site, nodata=-1, min_depth=0, max_depth=20, n_jobs=-1, chunk=100000):
    out = Path(out_dir) / f"Bathy_rast_{site}.tif"
    with rasterio.open(template_tif) as r:
        tr, crs, w, h = r.transform, r.crs, r.width, r.height
        tif_wkt = crs.to_wkt() if crs else None
    if tif_wkt is None: raise ValueError(f"No CRS in {template_tif}")
    flats, vals = [], []
    with fiona.open(shp) as src:
        dk = next((k for k in src.schema["properties"] if k.lower() == "depth"), None)
        shp_wkt = str(src.crs_wkt or src.crs)
        if dk is None: raise ValueError(f"No depth field in {shp}")
        if tif_wkt != shp_wkt: raise ValueError(f"CRS mismatch:\n{template_tif}\n{shp}")
        for f in src:
            g, v = f.get("geometry"), f["properties"].get(dk)
            if not g or v is None: continue
            try: v = abs(float(v))
            except: continue
            if not (min_depth <= v <= max_depth): continue
            pts = [g["coordinates"]] if g["type"] == "Point" else g["coordinates"] if g["type"] == "MultiPoint" else []
            for x, y in pts:
                rr, cc = rowcol(tr, x, y)
                if 0 <= rr < h and 0 <= cc < w:
                    flats.append(rr * w + cc); vals.append(v)
    s, c = np.zeros(h * w, np.float32), np.zeros(h * w, np.uint32)
    if flats:
        flats, vals = np.asarray(flats), np.asarray(vals, np.float32)
        def reduce_chunk(fa, va):
            u, inv = np.unique(fa, return_inverse=True)
            return u, np.bincount(inv, va).astype(np.float32), np.bincount(inv).astype(np.uint32)
        parts = [reduce_chunk(flats[i:i+chunk], vals[i:i+chunk]) for i in range(0, len(flats), chunk)] if n_jobs == 1 or len(flats) <= chunk \
            else Parallel(n_jobs=n_jobs)(delayed(reduce_chunk)(flats[i:i+chunk], vals[i:i+chunk]) for i in range(0, len(flats), chunk))
        for u, su, cu in parts:
            np.add.at(s, u, su); np.add.at(c, u, cu)
    arr = np.full((h, w), nodata, np.float32)
    c = c.reshape(h, w); m = c > 0
    arr[m] = s.reshape(h, w)[m] / c[m]
    prof = dict(driver="GTiff", height=h, width=w, count=1, dtype="float32", crs=crs, transform=tr,
                nodata=nodata, compress="ZSTD", predictor=2, BIGTIFF="IF_SAFER")
    with rasterio.open(out, "w", **prof) as dst: dst.write(arr, 1)
    return out
def make_1km_grid(template_rast, out_dir, site, cell_m=1000, seed=100):
    out = Path(out_dir) / f"1kmGrid_{site}.tif"
    with rasterio.open(template_rast) as src:
        tr, crs, w, h = src.transform, src.crs, src.width, src.height
    bx, by = max(1, round(cell_m / abs(tr.a))), max(1, round(cell_m / abs(tr.e)))
    gw, gh = (w + bx - 1) // bx, (h + by - 1) // by
    arr = np.repeat(np.repeat(np.random.default_rng(seed).integers(1, 6, (gh, gw), dtype=np.int8), by, 0), bx, 1)[:h, :w]
    prof = dict(driver="GTiff", height=h, width=w, count=1, dtype="int8", crs=crs, transform=tr,
                nodata=-1, compress="ZSTD", predictor=2, BIGTIFF="IF_SAFER")
    with rasterio.open(out, "w", **prof) as dst: dst.write(arr, 1)
    return out
def run_workflow_bathy_grid(tif_dir, bathy_dir, out_dir=None, site=None, min_depth=0, max_depth=20, cell_m=1000, seed=100, n_jobs=-1):
    tif_dir, bathy_dir = Path(tif_dir), Path(bathy_dir)
    out_dir = tif_dir if out_dir is None else Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    tifs = sorted(tif_dir.glob("*.tif"))
    if not tifs: raise FileNotFoundError(f"No tif files in {tif_dir}")
    site = site or tif_dir.name.split("_")[0].upper()
    shp = next((p for p in bathy_dir.glob(f"{site}_bathy_prj.shp")), None)
    if shp is None: raise FileNotFoundError(f"No shapefile found for {site} in {bathy_dir}")
    print(f"Site: {site}\nTemplate: {tifs[0].name}\nBathy: {shp.name}")
    bath = make_bath_rast(shp, tifs[0], out_dir, site, min_depth=min_depth, max_depth=max_depth, n_jobs=n_jobs)
    grid = make_1km_grid(bath, out_dir, site, cell_m=cell_m, seed=seed)
    print(f"Wrote:\n{bath}\n{grid}")
    return {"site": site, "bathy_rast": bath, "grid_1km": grid}

In [6]:
# === Cell 3: Make raster to deposite bathy predictions ===
def make_bathy_preds(tif_dir, out_dir=None, site=None, nodata=-100, dtype="float32"):
    tif_dir = Path(tif_dir)
    out_dir = tif_dir if out_dir is None else Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    site = site or tif_dir.name.split("_")[0].upper()
    bath = tif_dir / f"Bathy_rast_{site}.tif"
    grid = tif_dir / f"1kmGrid_{site}.tif"
    out_rf = out_dir / f"bathy_preds_{site}_RF.tif"
    out_st = out_dir / f"bathy_preds_{site}_stumpf.tif"
    if not bath.exists(): raise FileNotFoundError(f"Missing {bath.name}")
    if not grid.exists(): raise FileNotFoundError(f"Missing {grid.name}")
    src_tifs = sorted(p for p in tif_dir.glob("*.tif")
                      if not p.stem.lower().startswith(("bathy_rast_", "1kmgrid_", "bathy_preds_","mapbathy_preds_")))
    if not src_tifs: raise FileNotFoundError(f"No source tif files found in {tif_dir}")
    with rasterio.open(bath) as b, rasterio.open(grid) as g:
        if (b.width, b.height, b.transform, b.crs) != (g.width, g.height, g.transform, g.crs):
            raise ValueError("Bathy_rast and 1kmGrid do not match")
        m = b.read(1) == b.nodata if b.nodata is not None else np.zeros((b.height, b.width), bool)
        prof = b.profile.copy()
        prof.update(count=len(src_tifs), dtype=dtype, nodata=nodata, compress="ZSTD", predictor=2, BIGTIFF="IF_SAFER")
        fill = np.where(m, nodata, 0).astype(dtype)
        for out in (out_rf, out_st):
            with rasterio.open(out, "w", **prof) as dst:
                for i in range(1, len(src_tifs) + 1): dst.write(fill, i)
    return out_rf, out_st, len(src_tifs)
def run_workflow_bathy_pred_raster(tif_dir, out_dir=None, site=None, nodata=-100, dtype="float32"):
    tif_dir = Path(tif_dir)
    out_dir = tif_dir if out_dir is None else Path(out_dir)
    site = site or tif_dir.name.split("_")[0].upper()
    out_rf, out_st, nb = make_bathy_preds(tif_dir, out_dir=out_dir, site=site, nodata=nodata, dtype=dtype)
    print(f"Number of bands in each bathy_preds raster: {nb}")
    print(f"Wrote:\n{out_rf}\n{out_st}")
    return {"site": site, "bathy_preds_RF": out_rf, "bathy_preds_stumpf": out_st, "n_bands": nb}

In [7]:
# === Cell 4: predict all tifs and write bathy predictions===
def _ok_mask(bathy):
    with rasterio.open(bathy) as src: a,nod=src.read(1).astype(np.float32),src.nodata
    return np.isfinite(a) if nod is None else np.isfinite(a)&(a!=nod)
def _pad(win,pad,h,w):
    r0,c0=max(0,win.row_off-pad),max(0,win.col_off-pad); r1,c1=min(h,win.row_off+win.height+pad),min(w,win.col_off+win.width+pad)
    return Window(c0,r0,c1-c0,r1-r0),win.row_off-r0,win.col_off-c0
def _bad_mask(s2,nb=5,excl=15):
    with rasterio.open(s2) as src:
        h,w,nod=src.height,src.width,src.nodata; bad=np.zeros((h,w),bool); bands=range(1,min(nb,src.count)+1)
        for _,win in src.block_windows(1):
            a=src.read(list(bands),window=win).astype(np.float32); m=np.zeros((win.height,win.width),bool)
            for i,bi in enumerate(bands):
                mm=(src.read_masks(bi,window=win)==0)|~np.isfinite(a[i])
                if nod is not None and not (isinstance(nod,float) and np.isnan(nod)): mm|=a[i]==nod
                m|=mm
            bad[win.row_off:win.row_off+win.height,win.col_off:win.col_off+win.width]=m
    if excl>0: bad=binary_dilation(bad,structure=np.ones((3,3),bool),iterations=excl); bad[:excl,:]=bad[-excl:,:]=bad[:,:excl]=bad[:,-excl:]=True
    return bad
def _stumpf_ratio(b2, b3, n):
    n = float(n)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.log(n * b2) / np.log(n * b3)
def _fit_stumpf(b2, b3, y, n):
    b2 = np.asarray(b2, dtype=float)
    b3 = np.asarray(b3, dtype=float)
    y  = np.asarray(y,  dtype=float)
    m  = np.isfinite(b2) & np.isfinite(b3) & np.isfinite(y) & (b2 > 0) & (b3 > 0)
    r  = _stumpf_ratio(b2[m], b3[m], n)
    m2 = np.isfinite(r)
    slope, intercept = np.linalg.lstsq(np.c_[r[m2], np.ones(m2.sum())], y[m][m2], rcond=None)[0]
    return {"n": int(n), "slope": float(slope), "intercept": float(intercept)}
def _reset_preds_like_bathy(bath,preds_path,n_bands,nodata=-100,dtype="float32"):
    with rasterio.open(bath) as src:
        prof=src.profile.copy(); m=src.read(1)==src.nodata if src.nodata is not None else np.zeros((src.height,src.width),bool)
        fill=np.where(m,nodata,0).astype(dtype)
    prof.update(count=n_bands,dtype=dtype,nodata=nodata,compress="ZSTD",predictor=2,BIGTIFF="IF_SAFER")
    with rasterio.open(preds_path,"w",**prof) as dst:
        for i in range(1,n_bands+1): dst.write(fill,i)
def _assert_same_grid(a,b):
    with rasterio.open(a) as x, rasterio.open(b) as y:
        if (x.width,x.height,x.transform,x.crs)!=(y.width,y.height,y.transform,y.crs):
            raise ValueError(f"Grid mismatch:\n{Path(a).name}\n{Path(b).name}")
def _max_features_to_colsample(mf, n_feat):
    if mf == "auto": frac = 1.0
    elif mf == "sqrt": frac = np.sqrt(n_feat) / n_feat
    elif mf == "log2": frac = np.log2(n_feat) / n_feat
    else: raise ValueError(f"Unknown max_features: {mf}")
    return float(np.clip(frac, 1.0 / n_feat, 1.0))
def tune_xgbrf_trees_mae(df,group_col,y_col,feat_cols,n_trees_list=(30,100,300,1000),max_features_list=("auto","sqrt","log2"),seed=100):
    X=df[feat_cols].to_numpy(np.float32); y=df[y_col].to_numpy(np.float32); groups=df[group_col].to_numpy()
    n_feat=X.shape[1]
    uniq=np.unique(groups); best_mae=np.inf; best_t=None; best_mf=None; best_val=None; rows=[]
    for n_trees in n_trees_list:
        for mf in max_features_list:
            colsample=_max_features_to_colsample(mf,n_feat)
            pred_rows=[]
            for g in uniq:
                tr=groups!=g; va=groups==g
                if tr.sum()==0 or va.sum()==0: continue
                m=XGBRFRegressor(n_estimators=int(n_trees),colsample_bynode=colsample,n_jobs=-1,random_state=seed)
                m.fit(X[tr],y[tr]); pred=m.predict(X[va]).astype(np.float32,copy=False)
                for t,p in zip(y[va],pred): pred_rows.append({group_col:int(g),"True":float(t),"Predicted":float(p)})
            val_df=pd.DataFrame(pred_rows)
            mae=float(mean_absolute_error(val_df["True"],val_df["Predicted"])) if len(val_df) else np.inf
            rows.append({"n_estimators":int(n_trees),"max_features":mf,"colsample_bynode":colsample,"MAE":mae,"n_val":len(val_df)})
            if mae<best_mae: best_mae=mae; best_t=int(n_trees); best_mf=mf; best_val=val_df.copy()
    print(f"Best XGBRF: n_estimators={best_t} | max_features={best_mf} | MAE={best_mae:.4f}")
    best_colsample=_max_features_to_colsample(best_mf,n_feat)
    best_model=XGBRFRegressor(n_estimators=int(best_t),colsample_bynode=best_colsample,n_jobs=-1,random_state=seed); best_model.fit(X,y)
    return best_model,best_val,pd.DataFrame(rows),{"best_trees":best_t,"best_max_features":best_mf,"best_mae":best_mae}
def sample_rasts_with_avgs(s2:Path,bathy_rast:Path,grid_1km:Path,kernel_sizes=(3,5,7,9,11,13,15),max_bands=5,bathy_nodata=-1,min_depth=0,max_depth=20)->pd.DataFrame:
    ks=list(kernel_sizes); excl=int(max(ks))
    mn=min_depth if isinstance(min_depth,(int,np.integer,float,np.floating)) else None
    mx=max_depth if isinstance(max_depth,(int,np.integer,float,np.floating)) else None
    with rasterio.open(bathy_rast) as bsrc:
        depth=bsrc.read(1).astype(np.float32); bnod=bsrc.nodata if bsrc.nodata is not None else bathy_nodata
        depth_ok=(depth!=bnod)
        if mn is not None: depth_ok&=(depth>=mn)
        if mx is not None: depth_ok&=(depth<=mx)
        h,w=depth.shape
    with rasterio.open(s2) as ssrc:
        if (ssrc.height,ssrc.width)!=(h,w): raise ValueError(f"Shape mismatch bathy={h,w} vs s2={ssrc.height,ssrc.width}")
        nb=min(ssrc.count,int(max_bands)) if max_bands else ssrc.count; nod=ssrc.nodata
        bad=np.zeros((h,w),bool)
        for bi in range(1,nb+1):
            b=ssrc.read(bi).astype(np.float32)
            m=(ssrc.read_masks(bi)==0)|(~np.isfinite(b))
            if nod is not None and not (isinstance(nod,float) and np.isnan(nod)): m|=(b==nod)
            bad|=m; del b
        bad=binary_dilation(bad,structure=np.ones((3,3),bool),iterations=excl) if excl>0 else bad
        ok=depth_ok & (~bad); idx=np.flatnonzero(ok.ravel())
        cols=["group","depth"]+[f"B{i}" for i in range(1,nb+1)]+[f"B{i}_W{k}" for i in range(1,nb+1) for k in ks]
        print(f"Samples kept: {idx.size} (depth_ok={int(depth_ok.sum())}, bad={int(bad.sum())})")
        if idx.size==0: return pd.DataFrame(columns=cols)
        with rasterio.open(grid_1km) as gsrc: out={"group":gsrc.read(1).ravel()[idx].astype(np.int16),"depth":depth.ravel()[idx].astype(np.float32)}
        for bi in range(1,nb+1):
            b=ssrc.read(bi).astype(np.float32); vm=(ssrc.read_masks(bi)>0)&np.isfinite(b)
            if nod is not None and not (isinstance(nod,float) and np.isnan(nod)): vm&=(b!=nod)
            b0=np.where(vm,b,0).astype(np.float32,copy=False); vf=vm.astype(np.float32,copy=False)
            out[f"B{bi}"]=b.ravel()[idx]
            for k in ks:
                s=uniform_filter(b0,size=int(k),mode="mirror").astype(np.float32,copy=False)
                c=uniform_filter(vf,size=int(k),mode="mirror").astype(np.float32,copy=False)
                f=np.full((h,w),np.nan,np.float32); m=c>0; f[m]=s[m]/c[m]
                out[f"B{bi}_W{k}"]=f.ravel()[idx]
                del s,c,f; gc.collect()
            del b,vm,b0,vf; gc.collect()
    return pd.DataFrame(out)[cols]
def tune_stumpf_n_mae(df,group_col="group",y_col="depth",b2_col="B2",b3_col="B3",n_list=(500,1000,1500)):
    g,y=df[group_col].to_numpy(),df[y_col].to_numpy(np.float32); b2,b3=df[b2_col].to_numpy(np.float32),df[b3_col].to_numpy(np.float32)
    best_n,best_mae,best_val,rows=None,np.inf,None,[]
    for n in n_list:
        pred_rows=[]
        for k in sorted(np.unique(g)):
            tr,va=g!=k,g==k; p=_fit_stumpf(b2[tr],b3[tr],y[tr],n); r=_stumpf_ratio(b2[va],b3[va],n); m=np.isfinite(r)&np.isfinite(y[va])&(b2[va]>0)&(b3[va]>0)
            pred_rows += [{"group":int(k),"True":float(t),"Predicted":float(pr)} for t,pr in zip(y[va][m],p["slope"]*r[m]+p["intercept"])]
        val_df=pd.DataFrame(pred_rows); mae=np.mean(np.abs(val_df["True"]-val_df["Predicted"])) if len(val_df) else np.inf
        rows.append({"n":int(n),"MAE":float(mae),"n_val":len(val_df)})
        if mae<best_mae: best_n,best_mae,best_val=int(n),float(mae),val_df.copy()
    return best_val,pd.DataFrame(rows),{"best_n":best_n,"best_mae":best_mae}
def train_group_models(df,method,group_col="group",y_col="depth",feat_cols=None,best_trees=None,best_max_features=None,best_n=None,seed=100):
    g,y=df[group_col].to_numpy(),df[y_col].to_numpy(np.float32); out={}
    if method=="rf": X=df[feat_cols].to_numpy(np.float32); colsample=_max_features_to_colsample(best_max_features,X.shape[1])
    else: b2,b3=df["B2"].to_numpy(np.float32),df["B3"].to_numpy(np.float32)
    for k in sorted(np.unique(g)):
        tr=g!=k
        out[int(k)]=XGBRFRegressor(n_estimators=int(best_trees),colsample_bynode=colsample,n_jobs=-1,random_state=seed).fit(X[tr],y[tr]) if method=="rf" else _fit_stumpf(b2[tr],b3[tr],y[tr],best_n)
    return out
def _init_log(csv_path,cols):
    pd.DataFrame(columns=cols).to_csv(csv_path,index=False)
def _append_log(csv_path,row):
    pd.DataFrame([row]).to_csv(csv_path,mode="a",header=not os.path.exists(csv_path) or os.path.getsize(csv_path)==0,index=False)
def train_full_model(df,method,y_col="depth",feat_cols=None,best_trees=None,best_max_features=None,best_n=None,seed=100):
    y=df[y_col].to_numpy(np.float32)
    if method=="rf":
        X=df[feat_cols].to_numpy(np.float32); colsample=_max_features_to_colsample(best_max_features,X.shape[1])
        return XGBRFRegressor(n_estimators=int(best_trees),colsample_bynode=colsample,n_jobs=-1,random_state=seed).fit(X,y)
    return _fit_stumpf(df["B2"].to_numpy(np.float32),df["B3"].to_numpy(np.float32),y,best_n)
def predict_band_to_raster(s2,bathy,grid_tif,model_obj,bathy_preds_tif,band_num,method="rf",feat_cols=None,kernel_sizes=(3,5,7,9,11,13,15),max_bands=5,out_nodata=-100,ok=None,bad=None,use_groups=True):
    ks,pad,excl=list(kernel_sizes),max(kernel_sizes)//2,max(kernel_sizes); ok=_ok_mask(bathy) if ok is None else ok; bad=_bad_mask(s2,nb=max_bands,excl=excl) if bad is None else bad
    with rasterio.open(s2) as src, rasterio.open(grid_tif) as gsrc, rasterio.open(bathy_preds_tif,"r+") as dst:
        nod,bands=src.nodata,range(1,min(max_bands,src.count)+1)
        for _,win in src.block_windows(1):
            grid=gsrc.read(1,window=win); out=np.full((win.height,win.width),out_nodata,np.float32); base=ok[win.row_off:win.row_off+win.height,win.col_off:win.col_off+win.width]&~bad[win.row_off:win.row_off+win.height,win.col_off:win.col_off+win.width]
            if not base.any(): dst.write(out,band_num,window=win); continue
            if method=="stumpf":
                b2,b3=src.read(2,window=win).astype(np.float32),src.read(3,window=win).astype(np.float32); m2=(src.read_masks(2,window=win)>0)&np.isfinite(b2); m3=(src.read_masks(3,window=win)>0)&np.isfinite(b3)
                if nod is not None and not (isinstance(nod,float) and np.isnan(nod)): m2&=b2!=nod; m3&=b3!=nod
                models=model_obj.items() if use_groups else [(None,model_obj)]
                for g,m in models:
                    keep=(grid==g)&base&m2&m3&(b2>0)&(b3>0) if use_groups else base&m2&m3&(b2>0)&(b3>0)
                    r=_stumpf_ratio(b2,b3,m["n"]); keep&=np.isfinite(r); out[keep]=m["slope"]*r[keep]+m["intercept"]
            else:
                wpad,ri,ci=_pad(win,pad,src.height,src.width); rr,cc=slice(ri,ri+win.height),slice(ci,ci+win.width); a=src.read(list(bands),window=wpad).astype(np.float32); feat,valid={},[]
                for i,bi in enumerate(bands):
                    vm=(src.read_masks(bi,window=wpad)>0)&np.isfinite(a[i])
                    if nod is not None and not (isinstance(nod,float) and np.isnan(nod)): vm&=a[i]!=nod
                    valid.append(vm.astype(np.float32)); a[i]=np.where(vm,a[i],0); feat[f"B{bi}"]=a[i,rr,cc]
                for i,bi in enumerate(bands):
                    for k in ks:
                        s,c=uniform_filter(a[i],k,mode="mirror"),uniform_filter(valid[i],k,mode="mirror"); f=np.full_like(s,np.nan,np.float32); m=c>0; f[m]=s[m]/c[m]; feat[f"B{bi}_W{k}"]=f[rr,cc]
                cols=[feat[c].ravel() for c in feat_cols]
                if use_groups:
                    for g,m in model_obj.items():
                        keep=((grid==g)&base).ravel()
                        for v in cols: keep&=np.isfinite(v)
                        if keep.any(): out.ravel()[keep]=m.predict(np.column_stack([v[keep] for v in cols]).astype(np.float32)).astype(np.float32)
                else:
                    keep=base.ravel()
                    for v in cols: keep&=np.isfinite(v)
                    if keep.any(): out.ravel()[keep]=model_obj.predict(np.column_stack([v[keep] for v in cols]).astype(np.float32)).astype(np.float32)
            dst.write(out,band_num,window=win)
def run_workflow_predict(tif_dir,out_dir=None,site=None,kernel_sizes=(3,5,7,9,11,13,15),max_bands=5,min_depth=0,max_depth=20,n_trees_list=(30,100,300,1000),max_features_list=("auto","sqrt","log2"),n_list=(500,1000,1500),seed=100,out_nodata=-100):
    tif_dir=Path(tif_dir); out_dir=tif_dir if out_dir is None else Path(out_dir); out_dir.mkdir(parents=True,exist_ok=True); site=site or tif_dir.name.split("_")[0].upper()
    bath,grid=tif_dir/f"Bathy_rast_{site}.tif",tif_dir/f"1kmGrid_{site}.tif"
    preds_rf,preds_st=tif_dir/f"bathy_preds_{site}_RF.tif",tif_dir/f"bathy_preds_{site}_stumpf.tif"
    map_rf,map_st=tif_dir/f"MAPbathy_preds_{site}_RF.tif",tif_dir/f"MAPbathy_preds_{site}_stumpf.tif"
    rf_csv,st_csv=out_dir/f"RF_settings_log_{site}.csv",out_dir/f"STUMPF_settings_log_{site}.csv"; _init_log(rf_csv,["band","image_name","best_trees","best_max_features","MAE"]); _init_log(st_csv,["band","image_name","best_n","MAE"])
    src_tifs=sorted(p for p in tif_dir.glob("*.tif") if not p.stem.lower().startswith(("bathy_rast_","1kmgrid_","bathy_preds_","mapbathy_preds_","pred_xgbrf_","pred_stumpf_")))
    if not src_tifs: raise FileNotFoundError(f"No source tifs found in {tif_dir}")
    _reset_preds_like_bathy(bath,preds_rf,len(src_tifs),out_nodata); _reset_preds_like_bathy(bath,preds_st,len(src_tifs),out_nodata)
    shutil.copy2(preds_rf,map_rf); shutil.copy2(preds_st,map_st)
    for p in (preds_rf,preds_st,map_rf,map_st): _assert_same_grid(bath,p)
    ok=_ok_mask(bath)
    for band_num,tif in enumerate(src_tifs,1):
        df=sample_rasts_with_avgs(tif,bath,grid,kernel_sizes=kernel_sizes,max_bands=max_bands,min_depth=min_depth,max_depth=max_depth); feat_cols=[c for c in df.columns if c not in ("group","depth")]
        _,_,_,xmeta=tune_xgbrf_trees_mae(df,"group","depth",feat_cols,n_trees_list=n_trees_list,max_features_list=max_features_list,seed=seed); _,_,smeta=tune_stumpf_n_mae(df,"group","depth","B2","B3",n_list=n_list)
        rf_models=train_group_models(df,"rf",feat_cols=feat_cols,best_trees=xmeta["best_trees"],best_max_features=xmeta["best_max_features"],seed=seed); st_models=train_group_models(df,"stumpf",best_n=smeta["best_n"])
        rf_full=train_full_model(df,"rf",feat_cols=feat_cols,best_trees=xmeta["best_trees"],best_max_features=xmeta["best_max_features"],seed=seed); st_full=train_full_model(df,"stumpf",best_n=smeta["best_n"])
        bad=_bad_mask(tif,nb=max_bands,excl=max(kernel_sizes))
        predict_band_to_raster(tif,bath,grid,rf_models,preds_rf,band_num,"rf",feat_cols,kernel_sizes,max_bands,out_nodata,ok,bad,True)
        predict_band_to_raster(tif,bath,grid,st_models,preds_st,band_num,"stumpf",feat_cols,kernel_sizes,max_bands,out_nodata,ok,bad,True)
        predict_band_to_raster(tif,bath,grid,rf_full,map_rf,band_num,"rf",feat_cols,kernel_sizes,max_bands,out_nodata,ok,bad,False)
        predict_band_to_raster(tif,bath,grid,st_full,map_st,band_num,"stumpf",feat_cols,kernel_sizes,max_bands,out_nodata,ok,bad,False)
        _append_log(rf_csv,{"band":band_num,"image_name":tif.name,"best_trees":xmeta["best_trees"],"best_max_features":xmeta["best_max_features"],"MAE":xmeta["best_mae"]}); _append_log(st_csv,{"band":band_num,"image_name":tif.name,"best_n":smeta["best_n"],"MAE":smeta["best_mae"]})
        print(f"Updated band {band_num}/{len(src_tifs)} in preds + MAP rasters: {tif.name}")
    print(f"\nRF preds: {preds_rf}\nSTUMPF preds: {preds_st}\nMAP RF preds: {map_rf}\nMAP STUMPF preds: {map_st}\nRF log: {rf_csv}\nSTUMPF log: {st_csv}")
    return {"site":site,"n_bands":len(src_tifs),"bathy_preds_rf":preds_rf,"bathy_preds_stumpf":preds_st,"map_bathy_preds_rf":map_rf,"map_bathy_preds_stumpf":map_st,"rf_log_csv":rf_csv,"stumpf_log_csv":st_csv}

In [8]:
# === Cell 5: make mean/median/median_out3std rasters in target folders + csvs ===
def _make_mean(src_tif,out_mean,nodata=None):
    with rasterio.open(src_tif) as src:
        nodata=src.nodata if nodata is None else nodata
        prof=src.profile.copy(); prof.update(count=1,dtype="float32",nodata=nodata,compress="ZSTD",predictor=2,BIGTIFF="IF_SAFER")
        with rasterio.open(out_mean,"w",**prof) as d1:
            for _,win in src.block_windows(1):
                a=src.read(window=win).astype(np.float32)
                valid=np.isfinite(a) if nodata is None else np.isfinite(a)&(a!=nodata)
                a=np.where(valid,a,np.nan); n=valid.sum(axis=0)
                mean=np.full((win.height,win.width),nodata,np.float32)
                has=n>0
                if has.any(): mean[has]=np.nanmean(a[:,has],axis=0).astype(np.float32)
                d1.write(mean,1,window=win)
def _make_med(src_tif,out_med,nodata=None):
    with rasterio.open(src_tif) as src:
        nodata=src.nodata if nodata is None else nodata
        prof=src.profile.copy(); prof.update(count=1,dtype="float32",nodata=nodata,compress="ZSTD",predictor=2,BIGTIFF="IF_SAFER")
        with rasterio.open(out_med,"w",**prof) as d2:
            for _,win in src.block_windows(1):
                a=src.read(window=win).astype(np.float32)
                valid=np.isfinite(a) if nodata is None else np.isfinite(a)&(a!=nodata)
                a=np.where(valid,a,np.nan); n=valid.sum(axis=0)
                med=np.full((win.height,win.width),nodata,np.float32)
                has=n>0
                if has.any(): med[has]=np.nanmedian(a[:,has],axis=0).astype(np.float32)
                d2.write(med,1,window=win)
def _make_median_out3std(src_tif,out_median_out3std,k_std=3,nodata=None):
    with rasterio.open(src_tif) as src:
        nodata=src.nodata if nodata is None else nodata
        prof=src.profile.copy(); prof.update(count=1,dtype="float32",nodata=nodata,compress="ZSTD",predictor=2,BIGTIFF="IF_SAFER")
        with rasterio.open(out_median_out3std,"w",**prof) as d3:
            for _,win in src.block_windows(1):
                a=src.read(window=win).astype(np.float32)
                valid=np.isfinite(a) if nodata is None else np.isfinite(a)&(a!=nodata)
                a=np.where(valid,a,np.nan); n=valid.sum(axis=0)
                med3=np.full((win.height,win.width),nodata,np.float32)
                has=n>0
                if has.any():
                    b=a[:,has]; mu=np.nanmean(b,axis=0); sd=np.nanstd(b,axis=0)
                    keep=(b>=mu-k_std*sd)&(b<=mu+k_std*sd)
                    b2=np.where(keep,b,np.nan); has2=np.any(np.isfinite(b2),axis=0)
                    tmp=np.full(has.sum(),nodata,np.float32)
                    if has2.any(): tmp[has2]=np.nanmedian(b2[:,has2],axis=0).astype(np.float32)
                    med3[has]=tmp
                d3.write(med3,1,window=win)
def _sample_val_csv(pred_tif, bathy_tif, grid_tif, out_csv):
    with rasterio.open(bathy_tif) as b, rasterio.open(grid_tif) as g, rasterio.open(pred_tif) as p:
        if (b.width, b.height, b.transform, b.crs) != (g.width, g.height, g.transform, g.crs) or \
           (b.width, b.height, b.transform, b.crs) != (p.width, p.height, p.transform, p.crs):
            raise ValueError(f"Grid mismatch for {Path(pred_tif).name}")
        true, group, pred = b.read(1).astype(np.float32), g.read(1), p.read(1).astype(np.float32)
        m = np.isfinite(true) & np.isfinite(pred)
        if b.nodata is not None and not (isinstance(b.nodata, float) and np.isnan(b.nodata)): m &= true != b.nodata
        if p.nodata is not None and not (isinstance(p.nodata, float) and np.isnan(p.nodata)): m &= pred != p.nodata
        xs, ys = rasterio.transform.xy(b.transform, *np.where(m), offset="center")
        lons, lats = map(np.asarray, crs_transform(b.crs, "EPSG:4326", xs, ys))
        pd.DataFrame({"group": group[m].astype(np.int16), "TRUE": true[m], "Predicted": pred[m],
                      "lon": lons, "lat": lats}).to_csv(out_csv, index=False)
def run_workflow_median_export(tif_dir,site=None,k_std=3,log=None):
    tif_dir=Path(tif_dir); site=site or tif_dir.name.split("_")[0].upper()
    bath,grid=tif_dir/f"Bathy_rast_{site}.tif",tif_dir/f"1kmGrid_{site}.tif"
    d_mean,d_med,d_med3=tif_dir/f"pred_{site}_mean",tif_dir/f"pred_{site}_median",tif_dir/f"pred_{site}_median_out3std"
    d_mean.mkdir(parents=True,exist_ok=True); d_med.mkdir(parents=True,exist_ok=True); d_med3.mkdir(parents=True,exist_ok=True)
    stacks={"rf": tif_dir/f"bathy_preds_{site}_RF.tif",
        "stumpf": tif_dir/f"bathy_preds_{site}_stumpf.tif",
        "map_rf": tif_dir/f"MAPbathy_preds_{site}_RF.tif",
        "map_stumpf": tif_dir/f"MAPbathy_preds_{site}_stumpf.tif"}
    for p in stacks.values():
        if not p.exists(): raise FileNotFoundError(f"Missing {p.name}")
    outs={"mean_rf_tif": d_mean/f"mean_bathy_pred_{site}_RF.tif",
        "mean_stumpf_tif": d_mean/f"mean_bathy_pred_{site}_stumpf.tif",
        "median_rf_tif": d_med/f"med_bathy_pred_{site}_RF.tif",
        "median_stumpf_tif": d_med/f"med_bathy_pred_{site}_stumpf.tif",
        "median_out3std_rf_tif": d_med3/f"median_out3std_bathy_pred_{site}_RF.tif",
        "median_out3std_stumpf_tif": d_med3/f"median_out3std_bathy_pred_{site}_stumpf.tif",
        "MAP_mean_rf_tif": d_mean/f"MAP_mean_bathy_pred_{site}_RF.tif",
        "MAP_mean_stumpf_tif": d_mean/f"MAP_mean_bathy_pred_{site}_stumpf.tif",
        "MAP_median_rf_tif": d_med/f"MAP_med_bathy_pred_{site}_RF.tif",
        "MAP_median_stumpf_tif": d_med/f"MAP_med_bathy_pred_{site}_stumpf.tif",
        "MAP_median_out3std_rf_tif": d_med3/f"MAP_median_out3std_bathy_pred_{site}_RF.tif",
        "MAP_median_out3std_stumpf_tif": d_med3/f"MAP_median_out3std_bathy_pred_{site}_stumpf.tif"}
    t_mean_total=0.0; t_med_total=0.0; t_med3_total=0.0
    for key,src_key,mean_key,med_key,med3_key in [
        ("rf","rf","mean_rf_tif","median_rf_tif","median_out3std_rf_tif"),
        ("stumpf","stumpf","mean_stumpf_tif","median_stumpf_tif","median_out3std_stumpf_tif"),
        ("map_rf","map_rf","MAP_mean_rf_tif","MAP_median_rf_tif","MAP_median_out3std_rf_tif"),
        ("map_stumpf","map_stumpf","MAP_mean_stumpf_tif","MAP_median_stumpf_tif","MAP_median_out3std_stumpf_tif"),
    ]:
        t=time.perf_counter(); _make_mean(stacks[src_key],outs[mean_key]); dt=time.perf_counter()-t; t_mean_total+=dt
        if log is not None: log.append(f"  _make_mean ({key}): {dt:.2f} seconds")
        t=time.perf_counter(); _make_med(stacks[src_key],outs[med_key]); dt=time.perf_counter()-t; t_med_total+=dt
        if log is not None: log.append(f"  _make_med ({key}): {dt:.2f} seconds")
        t=time.perf_counter(); _make_median_out3std(stacks[src_key],outs[med3_key],k_std=k_std); dt=time.perf_counter()-t; t_med3_total+=dt
        if log is not None: log.append(f"  _make_median_out3std ({key}): {dt:.2f} seconds")
    if log is not None: log.append(f"  [mean total]: {t_mean_total:.2f} seconds"); log.append(f"  [median total]: {t_med_total:.2f} seconds"); log.append(f"  [median_out3std total]: {t_med3_total:.2f} seconds")
    mean_rf_csv=d_mean/"XGBRF_val_predictions.csv"
    mean_st_csv=d_mean/"STUMPF_val_predictions.csv"
    med_rf_csv=d_med/"XGBRF_val_predictions.csv"
    med_st_csv=d_med/"STUMPF_val_predictions.csv"
    med3_rf_csv=d_med3/"XGBRF_val_predictions.csv"
    med3_st_csv=d_med3/"STUMPF_val_predictions.csv"
    _sample_val_csv(outs["mean_rf_tif"],bath,grid,mean_rf_csv)
    _sample_val_csv(outs["mean_stumpf_tif"],bath,grid,mean_st_csv)
    _sample_val_csv(outs["median_rf_tif"],bath,grid,med_rf_csv)
    _sample_val_csv(outs["median_stumpf_tif"],bath,grid,med_st_csv)
    _sample_val_csv(outs["median_out3std_rf_tif"],bath,grid,med3_rf_csv)
    _sample_val_csv(outs["median_out3std_stumpf_tif"],bath,grid,med3_st_csv)
    print(f"Wrote:\n{d_mean}\n{d_med}\n{d_med3}")
    return {"site":site,"mean_dir":d_mean,"median_dir":d_med,"median_out3std_dir":d_med3,**outs,
            "mean_rf_csv":mean_rf_csv,"mean_stumpf_csv":mean_st_csv,
            "median_rf_csv":med_rf_csv,"median_stumpf_csv":med_st_csv,
            "median_out3std_rf_csv":med3_rf_csv,"median_out3std_stumpf_csv":med3_st_csv}

In [9]:
input_dir=r"E:\GIS\SDB_Compare\Cape Cod\CC_TIFS"
input_dir=Path(input_dir); log=[]; start_dt=datetime.now(); t_all=time.perf_counter()
log.append(f"Start time: {start_dt:%Y-%m-%d %H:%M:%S}")
for name, fn, kwargs in [
    ("run_workflow_bathy_grid", run_workflow_bathy_grid, dict(tif_dir=input_dir, bathy_dir=r"E:\GIS\SDB_Compare\Bathy_shps", out_dir=input_dir)),
    ("run_workflow_bathy_pred_raster", run_workflow_bathy_pred_raster, dict(tif_dir=input_dir, out_dir=input_dir)),
    ("run_workflow_predict", run_workflow_predict, dict(tif_dir=input_dir, out_dir=input_dir)),
    ("run_workflow_median_export", run_workflow_median_export, dict(tif_dir=input_dir, log=log)),
]:
    t=time.perf_counter(); fn(**kwargs); log.append(f"{name}: {time.perf_counter()-t:.2f} seconds")
end_dt=datetime.now(); total=time.perf_counter()-t_all
log += [f"End time: {end_dt:%Y-%m-%d %H:%M:%S}", f"Total duration: {int(total//3600):02d}:{int((total%3600)//60):02d}:{total%60:05.2f}"]
log_path=input_dir/"workflow_timing_log.txt"
log_path.write_text("\n".join(log), encoding="utf-8")
print("\n".join(log), f"\n\nTiming log written to: {log_path}")

Site: CC
Template: S2A_MSIL2A_20160624T153602_N0500_R111_T19TCG_20231030T222159_plusSCL_masked.tif
Bathy: CC_bathy_prj.shp
Wrote:
E:\GIS\SDB_Compare\Cape Cod\CC_TIFS\Bathy_rast_CC.tif
E:\GIS\SDB_Compare\Cape Cod\CC_TIFS\1kmGrid_CC.tif
Number of bands in each bathy_preds raster: 31
Wrote:
E:\GIS\SDB_Compare\Cape Cod\CC_TIFS\bathy_preds_CC_RF.tif
E:\GIS\SDB_Compare\Cape Cod\CC_TIFS\bathy_preds_CC_stumpf.tif
Samples kept: 486223 (depth_ok=499050, bad=19124)
Best XGBRF: n_estimators=1000 | max_features=sqrt | MAE=0.5386
Updated band 1/31 in preds + MAP rasters: S2A_MSIL2A_20160624T153602_N0500_R111_T19TCG_20231030T222159_plusSCL_masked.tif
Samples kept: 496997 (depth_ok=499050, bad=3972)
Best XGBRF: n_estimators=100 | max_features=sqrt | MAE=0.7340
Updated band 2/31 in preds + MAP rasters: S2A_MSIL2A_20160803T153602_N0500_R111_T19TCG_20231030T195708_plusSCL_masked.tif
Samples kept: 483854 (depth_ok=499050, bad=18795)
Best XGBRF: n_estimators=300 | max_features=sqrt | MAE=0.5896
Updated ban

In [10]:
input_dir=r"E:\GIS\SDB_Compare\Ungava Bay\UB_TIFS"
input_dir=Path(input_dir); log=[]; start_dt=datetime.now(); t_all=time.perf_counter()
log.append(f"Start time: {start_dt:%Y-%m-%d %H:%M:%S}")
for name, fn, kwargs in [
    ("run_workflow_bathy_grid", run_workflow_bathy_grid, dict(tif_dir=input_dir, bathy_dir=r"E:\GIS\SDB_Compare\Bathy_shps", out_dir=input_dir)),
    ("run_workflow_bathy_pred_raster", run_workflow_bathy_pred_raster, dict(tif_dir=input_dir, out_dir=input_dir)),
    ("run_workflow_predict", run_workflow_predict, dict(tif_dir=input_dir, out_dir=input_dir)),
    ("run_workflow_median_export", run_workflow_median_export, dict(tif_dir=input_dir, log=log)),
]:
    t=time.perf_counter(); fn(**kwargs); log.append(f"{name}: {time.perf_counter()-t:.2f} seconds")
end_dt=datetime.now(); total=time.perf_counter()-t_all
log += [f"End time: {end_dt:%Y-%m-%d %H:%M:%S}", f"Total duration: {int(total//3600):02d}:{int((total%3600)//60):02d}:{total%60:05.2f}"]
log_path=input_dir/"workflow_timing_log.txt"
log_path.write_text("\n".join(log), encoding="utf-8")
print("\n".join(log), f"\n\nTiming log written to: {log_path}")

Site: UB
Template: S2A_MSIL2A_20160726T161352_N0500_R140_T19VDG_20231002T154004_plusSCL_masked.tif
Bathy: UB_bathy_prj.shp
Wrote:
E:\GIS\SDB_Compare\Ungava Bay\UB_TIFS\Bathy_rast_UB.tif
E:\GIS\SDB_Compare\Ungava Bay\UB_TIFS\1kmGrid_UB.tif
Number of bands in each bathy_preds raster: 18
Wrote:
E:\GIS\SDB_Compare\Ungava Bay\UB_TIFS\bathy_preds_UB_RF.tif
E:\GIS\SDB_Compare\Ungava Bay\UB_TIFS\bathy_preds_UB_stumpf.tif
Samples kept: 91408 (depth_ok=94626, bad=213296)
Best XGBRF: n_estimators=1000 | max_features=sqrt | MAE=2.1988
Updated band 1/18 in preds + MAP rasters: S2A_MSIL2A_20160726T161352_N0500_R140_T19VDG_20231002T154004_plusSCL_masked.tif
Samples kept: 94626 (depth_ok=94626, bad=8956)
Best XGBRF: n_estimators=300 | max_features=sqrt | MAE=2.0940
Updated band 2/18 in preds + MAP rasters: S2A_MSIL2A_20160802T155912_N0500_R097_T19VDG_20231022T045928_plusSCL_masked.tif
Samples kept: 92902 (depth_ok=94626, bad=47617)
Best XGBRF: n_estimators=300 | max_features=auto | MAE=2.1491
Updated 

In [11]:
# === Cell 6: run ===
input_dir=r"E:\GIS\SDB_Compare\Corsica\COR_TIFS"
input_dir=Path(input_dir); log=[]; start_dt=datetime.now(); t_all=time.perf_counter()
log.append(f"Start time: {start_dt:%Y-%m-%d %H:%M:%S}")
for name, fn, kwargs in [
    ("run_workflow_bathy_grid", run_workflow_bathy_grid, dict(tif_dir=input_dir, bathy_dir=r"E:\GIS\SDB_Compare\Bathy_shps", out_dir=input_dir)),
    ("run_workflow_bathy_pred_raster", run_workflow_bathy_pred_raster, dict(tif_dir=input_dir, out_dir=input_dir)),
    ("run_workflow_predict", run_workflow_predict, dict(tif_dir=input_dir, out_dir=input_dir)),
    ("run_workflow_median_export", run_workflow_median_export, dict(tif_dir=input_dir, log=log)),
]:
    t=time.perf_counter(); fn(**kwargs); log.append(f"{name}: {time.perf_counter()-t:.2f} seconds")
end_dt=datetime.now(); total=time.perf_counter()-t_all
log += [f"End time: {end_dt:%Y-%m-%d %H:%M:%S}", f"Total duration: {int(total//3600):02d}:{int((total%3600)//60):02d}:{total%60:05.2f}"]
log_path=input_dir/"workflow_timing_log.txt"
log_path.write_text("\n".join(log), encoding="utf-8")
print("\n".join(log), f"\n\nTiming log written to: {log_path}")

Site: COR
Template: S2A_MSIL2A_20170216T102101_N0500_R065_T32TNL_20231025T172455_plusSCL_masked.tif
Bathy: COR_bathy_prj.shp
Wrote:
E:\GIS\SDB_Compare\Corsica\COR_TIFS\Bathy_rast_COR.tif
E:\GIS\SDB_Compare\Corsica\COR_TIFS\1kmGrid_COR.tif
Number of bands in each bathy_preds raster: 100
Wrote:
E:\GIS\SDB_Compare\Corsica\COR_TIFS\bathy_preds_COR_RF.tif
E:\GIS\SDB_Compare\Corsica\COR_TIFS\bathy_preds_COR_stumpf.tif
Samples kept: 382550 (depth_ok=428502, bad=843884)
Best XGBRF: n_estimators=1000 | max_features=auto | MAE=2.6950
Updated band 1/100 in preds + MAP rasters: S2A_MSIL2A_20170216T102101_N0500_R065_T32TNL_20231025T172455_plusSCL_masked.tif
Samples kept: 404795 (depth_ok=428502, bad=64096)
Best XGBRF: n_estimators=1000 | max_features=auto | MAE=2.2718
Updated band 2/100 in preds + MAP rasters: S2A_MSIL2A_20170308T102021_N0500_R065_T32TNL_20231024T113345_plusSCL_masked.tif
Samples kept: 347127 (depth_ok=428502, bad=154080)
Best XGBRF: n_estimators=100 | max_features=sqrt | MAE=1.541

In [12]:
input_dir=r"E:\GIS\SDB_Compare\Key West\KW_TIFS"
input_dir=Path(input_dir); log=[]; start_dt=datetime.now(); t_all=time.perf_counter()
log.append(f"Start time: {start_dt:%Y-%m-%d %H:%M:%S}")
for name, fn, kwargs in [
    ("run_workflow_bathy_grid", run_workflow_bathy_grid, dict(tif_dir=input_dir, bathy_dir=r"E:\GIS\SDB_Compare\Bathy_shps", out_dir=input_dir)),
    ("run_workflow_bathy_pred_raster", run_workflow_bathy_pred_raster, dict(tif_dir=input_dir, out_dir=input_dir)),
    ("run_workflow_predict", run_workflow_predict, dict(tif_dir=input_dir, out_dir=input_dir)),
    ("run_workflow_median_export", run_workflow_median_export, dict(tif_dir=input_dir, log=log)),
]:
    t=time.perf_counter(); fn(**kwargs); log.append(f"{name}: {time.perf_counter()-t:.2f} seconds")
end_dt=datetime.now(); total=time.perf_counter()-t_all
log += [f"End time: {end_dt:%Y-%m-%d %H:%M:%S}", f"Total duration: {int(total//3600):02d}:{int((total%3600)//60):02d}:{total%60:05.2f}"]
log_path=input_dir/"workflow_timing_log.txt"
log_path.write_text("\n".join(log), encoding="utf-8")
print("\n".join(log), f"\n\nTiming log written to: {log_path}")

Site: KW
Template: S2A_MSIL2A_20171016T160241_N0500_R097_T17RMH_20230824T064102_plusSCL_masked.tif
Bathy: KW_bathy_prj.shp
Wrote:
E:\GIS\SDB_Compare\Key West\KW_TIFS\Bathy_rast_KW.tif
E:\GIS\SDB_Compare\Key West\KW_TIFS\1kmGrid_KW.tif
Number of bands in each bathy_preds raster: 57
Wrote:
E:\GIS\SDB_Compare\Key West\KW_TIFS\bathy_preds_KW_RF.tif
E:\GIS\SDB_Compare\Key West\KW_TIFS\bathy_preds_KW_stumpf.tif
Samples kept: 1009385 (depth_ok=1044854, bad=38291)
Best XGBRF: n_estimators=300 | max_features=sqrt | MAE=0.2478
Updated band 1/57 in preds + MAP rasters: S2A_MSIL2A_20171016T160241_N0500_R097_T17RMH_20230824T064102_plusSCL_masked.tif
Samples kept: 1037878 (depth_ok=1044854, bad=10063)
Best XGBRF: n_estimators=100 | max_features=sqrt | MAE=0.2425
Updated band 2/57 in preds + MAP rasters: S2A_MSIL2A_20171105T160441_N0500_R097_T17RMH_20231014T060528_plusSCL_masked.tif
Samples kept: 961980 (depth_ok=1044854, bad=87811)
Best XGBRF: n_estimators=1000 | max_features=sqrt | MAE=0.1820
Updat

In [4]:
### New methods
# === Cell 7: lowvar50 / highcomplete50 median composites ===
import re
LIST_DIR = r"E:\GIS\SDB_Compare\Product_comp_img_lists"
METHODS = ("lowvar50", "highcomplete50")
MAP_CSVS = False
SKIP_PREFIXES = ("bathy_rast_", "1kmgrid_", "bathy_preds_", "mapbathy_preds_", "pred_xgbrf_", "pred_stumpf_")

def _fmt_hms(s):
    return f"{int(s//3600):02d}:{int((s%3600)//60):02d}:{s%60:05.2f}"
def _read_img_list(txt):
    lines = Path(txt).read_text(encoding="utf-8").splitlines()
    names = [l.strip() for l in lines[next(i for i, l in enumerate(lines) if l.strip().lower() == "images:") + 1:] if l.strip()]
    n = next((int(re.search(r"\d+", l).group()) for l in lines if l.lower().startswith("n_images")), None)
    if n is not None and n != len(names): raise ValueError(f"{Path(txt).name}: n_images={n} but {len(names)} names listed")
    return names
def _parse_timing_log(path):
    out = {}
    for l in Path(path).read_text(encoding="utf-8").splitlines():
        m = re.match(r"^(run_workflow_\w+): ([\d.]+) seconds", l)
        if m: out[m.group(1)] = float(m.group(2))
    return out
def _make_med_subset(src_tif, out_tif, band_idx, nodata=None):
    with rasterio.open(src_tif) as src:
        nodata = src.nodata if nodata is None else nodata
        prof = src.profile.copy(); prof.update(count=1, dtype="float32", nodata=nodata, compress="ZSTD", predictor=2, BIGTIFF="IF_SAFER")
        with rasterio.open(out_tif, "w", **prof) as dst:
            for _, win in src.block_windows(1):
                a = src.read(band_idx, window=win).astype(np.float32)
                valid = np.isfinite(a) if nodata is None else np.isfinite(a) & (a != nodata)
                a = np.where(valid, a, np.nan); has = valid.any(axis=0)
                med = np.full((win.height, win.width), nodata, np.float32)
                if has.any(): med[has] = np.nanmedian(a[:, has], axis=0).astype(np.float32)
                dst.write(med, 1, window=win)
def run_workflow_median_subset(tif_dir, list_dir=LIST_DIR, methods=METHODS, site=None, map_csvs=MAP_CSVS):
    tif_dir = Path(tif_dir); site = site or tif_dir.name.split("_")[0].upper()
    bath, grid = tif_dir / f"Bathy_rast_{site}.tif", tif_dir / f"1kmGrid_{site}.tif"
    stacks = {"RF": tif_dir / f"bathy_preds_{site}_RF.tif", "stumpf": tif_dir / f"bathy_preds_{site}_stumpf.tif",
              "MAP_RF": tif_dir / f"MAPbathy_preds_{site}_RF.tif", "MAP_stumpf": tif_dir / f"MAPbathy_preds_{site}_stumpf.tif"}
    for p in stacks.values():
        if not p.exists(): raise FileNotFoundError(f"Missing {p.name}")
    src_tifs = sorted(p for p in tif_dir.glob("*.tif") if not p.stem.lower().startswith(SKIP_PREFIXES))
    stem_to_band = {p.stem: i for i, p in enumerate(src_tifs, 1)}; n_total = len(src_tifs)
    t = _parse_timing_log(tif_dir / "workflow_timing_log.txt")
    t_grid, t_pred_rast, t_predict = t["run_workflow_bathy_grid"], t["run_workflow_bathy_pred_raster"], t["run_workflow_predict"]
    res = {}
    for method in methods:
        t0 = time.perf_counter()
        names = _read_img_list(Path(list_dir) / f"{site}_{method}_median.txt")
        missing = [n for n in names if n not in stem_to_band]
        if missing: raise ValueError(f"{site} {method}: {len(missing)} images not found in {tif_dir.name}, e.g. {missing[0]}")
        band_idx = [stem_to_band[n] for n in names]
        out_dir = tif_dir / f"pred_{site}_{method}_median"; out_dir.mkdir(parents=True, exist_ok=True)
        outs = {k: out_dir / f"{'MAP_' if k.startswith('MAP') else ''}med_{method}_bathy_pred_{site}_{k.replace('MAP_', '')}.tif" for k in stacks}
        for k, src in stacks.items(): _make_med_subset(src, outs[k], band_idx)
        csvs = {"RF": "XGBRF_val_predictions.csv", "stumpf": "STUMPF_val_predictions.csv"}
        if map_csvs: csvs.update({"MAP_RF": "MAP_XGBRF_val_predictions.csv", "MAP_stumpf": "MAP_STUMPF_val_predictions.csv"})
        for k, name in csvs.items(): _sample_val_csv(outs[k], bath, grid, out_dir / name)
        t_med = time.perf_counter() - t0
        est = t_grid + t_pred_rast + t_predict * len(names) / n_total + t_med
        (out_dir / f"estimated_time_{site}_{method}.txt").write_text("\n".join([
            f"site: {site}", f"method: {method}", f"images used: {len(names)} / {n_total}",
            f"run_workflow_bathy_grid: {t_grid:.2f} seconds", f"run_workflow_bathy_pred_raster: {t_pred_rast:.2f} seconds",
            f"run_workflow_predict (full): {t_predict:.2f} seconds",
            f"run_workflow_predict (scaled {len(names)}/{n_total}): {t_predict * len(names) / n_total:.2f} seconds",
            f"median composite + csv time: {t_med:.2f} seconds",
            f"estimated total: {est:.2f} seconds ({_fmt_hms(est)})"]), encoding="utf-8")
        print(f"{site} {method}: {len(names)}/{n_total} images -> {out_dir} | est {_fmt_hms(est)}")
        res[method] = {"dir": out_dir, "n_images": len(names), "est_seconds": est}
    return res

In [9]:
# === Cell 8: run subset medians for all sites ===
SITE_DIRS = [r"E:\GIS\SDB_Compare\Cape Cod\CC_TIFS", r"E:\GIS\SDB_Compare\Ungava Bay\UB_TIFS",
             r"E:\GIS\SDB_Compare\Corsica\COR_TIFS", r"E:\GIS\SDB_Compare\Key West\KW_TIFS"]
for d in SITE_DIRS: run_workflow_median_subset(d)

CC lowvar50: 16/31 images -> E:\GIS\SDB_Compare\Cape Cod\CC_TIFS\pred_CC_lowvar50_median | est 01:43:35.88
CC highcomplete50: 16/31 images -> E:\GIS\SDB_Compare\Cape Cod\CC_TIFS\pred_CC_highcomplete50_median | est 01:43:36.90
UB lowvar50: 9/18 images -> E:\GIS\SDB_Compare\Ungava Bay\UB_TIFS\pred_UB_lowvar50_median | est 00:14:31.46
UB highcomplete50: 9/18 images -> E:\GIS\SDB_Compare\Ungava Bay\UB_TIFS\pred_UB_highcomplete50_median | est 00:14:30.60
COR lowvar50: 50/100 images -> E:\GIS\SDB_Compare\Corsica\COR_TIFS\pred_COR_lowvar50_median | est 05:07:04.14
COR highcomplete50: 50/100 images -> E:\GIS\SDB_Compare\Corsica\COR_TIFS\pred_COR_highcomplete50_median | est 05:06:53.06
KW lowvar50: 29/57 images -> E:\GIS\SDB_Compare\Key West\KW_TIFS\pred_KW_lowvar50_median | est 07:17:58.88
KW highcomplete50: 29/57 images -> E:\GIS\SDB_Compare\Key West\KW_TIFS\pred_KW_highcomplete50_median | est 07:17:52.55
